# Stage 0: Setup

In [1]:
### Imports
### System
import sys, os

### I/O
import h5py
import tables_io

### Operations
import numpy as np

### RAIL
from rail.core.data import DataStore
from rail.core.stage import RailStage
from rail.core.data import PqHandle

sys.path.insert(0, "/global/homes/s/sajkov/rail_umap/src/degrading")
from MultiSurveyErrorModel import MultiSurveyErrorModel

from rail.estimation.algos.lephare import LephareInformer, LephareEstimator
import lephare as lp

sys.path.insert(0, "/global/homes/s/sajkov/rail_umap/src/estimation")
from UMAPEstimator import UMAPEstimator

### Set the date, start the timer
import time
date = time.strftime('%d%b%y', time.localtime())

LEPHAREDIR is being set to the default cache directory:
/global/homes/s/sajkov/.cache/lephare/data
More than 1Gb may be written there.
LEPHAREWORK is being set to the default cache directory:
/global/homes/s/sajkov/.cache/lephare/work
Default work cache is already linked. 
This is linked to the run directory:
/global/homes/s/sajkov/.cache/lephare/runs/20260702T160835


In [2]:
### for testing only
import matplotlib.pyplot as plt
plt.style.use("/global/homes/s/sajkov/umap_nz_cal.mplstyle")

In [3]:
### Initialize random state
seed = 42
rng = np.random.default_rng(seed = seed)

In [28]:
### Specify outputs directory
outputs_directory = f"/pscratch/sd/s/sajkov/analysis_pipeline/runs/{date}"
if not os.path.exists(outputs_directory):
    os.makedirs(outputs_directory)

# Stage 1: Create datasets


In [5]:
### Specify path to noiseless catalog and redshifts
noiseless_catalog_filepath = "/pscratch/sd/s/sajkov/data/integrated_catalog_23apr26.pq"
redshifts_filepath = "/pscratch/sd/s/sajkov/data/mock_catalog_Ch1_26.h5"

In [6]:
DATASET_popCosmos_full   = tables_io.read(noiseless_catalog_filepath)
REDSHIFTS_popCosmos_full = h5py.File(redshifts_filepath)['sps_parameters'][:, -1]

column_list None


In [7]:
### Number of pop-cosmos sources to use in analysis
data_cut = 100_000

### Fraction of deep field versus WFD photometry
deep_field_frac = 0.2

### Randomize full dataset indices, take `deep_field_frac` to be the deep field, let the rest be WFD
RANDIDX_popCosmos_full = rng.choice(np.arange(len(DATASET_popCosmos_full)), len(DATASET_popCosmos_full), replace = False)
RANDIDX_popCosmos_cut  = RANDIDX_popCosmos_full[:data_cut]

deep_field_cut = int(deep_field_frac * len(RANDIDX_popCosmos_cut))
IDX_popCosmos_DeepField = RANDIDX_popCosmos_cut[:deep_field_cut]
IDX_popCosmos_WideFastDeep = RANDIDX_popCosmos_cut[deep_field_cut:]

In [8]:
### 5-sigma limiting depths ------------------
### LSST: median values for COSMOS deep field from https://usdf-maf.slac.stanford.edu/summaryStats?runId=5#Basics_Coadd%20M5
### Roman from https://github.com/jfcrenshaw/photerr/blob/a014b39729ddde3daf80be2dbe82f5a7f958882c/photerr/roman.py#L120-L126
### HSC Niji (60 min exposure) from https://sites.google.com/view/hsc-mb-survey3/filter-specification?authuser=0 
###
### Acessed June 3, 2026
### --------------------------------------------

M5_DEPTHS_DeepField = {'LSST_u'    : 27.74,
                      'LSST_g'    : 28.69,
                      'LSST_r'    : 28.88,
                      'LSST_i'    : 28.96,
                      'LSST_z'    : 28.26,
                      'LSST_y'    : 26.63,
                      'Roman_F062': 27.7,
                      'Roman_F087': 27.7,
                      'Roman_F106': 27.6,
                      'Roman_F129': 27.5,
                      'Roman_F158': 27.0,
                      'Roman_F184': 25.9,
                      'Roman_F213': 28.3,
                      'HSC_MB_00' : 26.41,
                      'HSC_MB_01' : 26.51,
                      'HSC_MB_02' : 26.45,
                      'HSC_MB_03' : 26.69,
                      'HSC_MB_04' : 26.93,
                      'HSC_MB_05' : 26.62,
                      'HSC_MB_06' : 26.26,
                      'HSC_MB_07' : 26.02,
                      'HSC_MB_08' : 26.07,
                      'HSC_MB_09' : 26.00,
                      'HSC_MB_10' : 26.06,
                      'HSC_MB_11' : 25.52,
                      'HSC_MB_12' : 25.58,
                      'HSC_MB_13' : 25.43,
                      'HSC_MB_14' : 25.15,
                      'HSC_MB_15' : 24.79}


### 5-sigma limiting depths for WideFastDeep from https://usdf-maf.slac.stanford.edu/summaryStats?runId=5#Basics_Coadd%20M5
### Column `DD:WFD CoaddM5`
M5_DEPTHS_WideFastDeep = {'LSST_u'    : 25.61,
                          'LSST_g'    : 26.90,
                          'LSST_r'    : 26.87,
                          'LSST_i'    : 26.43,
                          'LSST_z'    : 25.73,
                          'LSST_y'    : 24.79}

### Get list of bands
BANDS_DeepField = list(M5_DEPTHS_DeepField.keys())
BANDS_WideFastDeep = list(M5_DEPTHS_WideFastDeep.keys())

In [9]:
### Select needed bands and pick out needed sources
PHOTOMETRY_DeepField_noiseless    = DATASET_popCosmos_full[BANDS_DeepField].iloc[IDX_popCosmos_DeepField]
PHOTOMETRY_WideFastDeep_noiseless = DATASET_popCosmos_full[BANDS_WideFastDeep].iloc[IDX_popCosmos_WideFastDeep]

### Same as above, for redshifts
REDSHIFTS_DeepField    = REDSHIFTS_popCosmos_full[IDX_popCosmos_DeepField]
REDSHIFTS_WideFastDeep = REDSHIFTS_popCosmos_full[IDX_popCosmos_WideFastDeep]

## Apply noise

In [10]:
### Noising parameters

nYrObs     = 1 # one-year depths
nVisYr     = 1 # one visit/yr (i.e., no co-adds)
gamma      = 0.04
sigLim     = 0 # 
inputType  = 'pogson' # input pogson magnitudes (AB)
outputType = 'asinh'  # output asinh magnitudes

seed = 42

### DP 1.1: Deep, multi-band, medium-band photometry + degraded LSST photometry

In [11]:
noisyPhotometryPath_DeepField = f"{outputs_directory}/PHOTOMETRY_DeepField_noisy_{date}.pq"

getNoisyDeepFieldPhotometry = MultiSurveyErrorModel.make_stage(
    name = "getNoisyDeepFieldPhotometry",
    
    inputType  = inputType,
    outputType = outputType,

    noisy_catalog     = noisyPhotometryPath_DeepField,
    
    m5     = M5_DEPTHS_DeepField,
    bands  = BANDS_DeepField,
    nYrObs = nYrObs,
    nVisYr = nVisYr,
    gamma  = gamma,
    sigLim = sigLim,
    
    seed = seed
)

getNoisyDeepFieldPhotometry.set_data("noiseless_catalog", PHOTOMETRY_DeepField_noiseless) 
getNoisyDeepFieldPhotometry.run()
getNoisyDeepFieldPhotometry.get_handle("noisy_catalog").write()
getNoisyDeepFieldPhotometry.finalize()

2026-07-03 16:56:02 [info     ] Inserting handle into data store.  noiseless_catalog: None, getNoisyDeepFieldPhotometry
2026-07-03 16:56:02 [info     ] Inserting handle into data store.  noisy_catalog_getNoisyDeepFieldPhotometry: /pscratch/sd/s/sajkov/analysis_pipeline_outputs/03Jul26/inprogress_PHOTOMETRY_DeepField_noisy_03Jul26.pq, getNoisyDeepFieldPhotometry


### *(((RETURN TO THIS)))* DP 1.2: same as DP 1.1, but with WideFastDeep 5 sigma depths

In [12]:
# degradedPhotometryPath_DeepField = f"{outputs_directory}/PHOTOMETRY_DeepField_degraded_{date}.pq"

# degradeDeepFieldPhotometry = MultiSurveyErrorModel.make_stage(
#     name = "degradeDeepFieldPhotometry",
    
#     inputType  = inputType,
#     outputType = outputType,

#     noisy_catalog     = degradedPhotometryPath_DeepField,
    
#     m5     = M5_DEPTHS_WideFastDeep,
#     bands  = BANDS_WideFastDeep,
#     nYrObs = nYrObs,
#     nVisYr = nVisYr,
#     gamma  = gamma,
#     sigLim = sigLim,
    
#     seed = seed
# )

# degradeDeepFieldPhotometry.set_data("noiseless_catalog", PHOTOMETRY_DeepField_noiseless) 
# degradeDeepFieldPhotometry.run()
# degradeDeepFieldPhotometry.get_handle("noisy_catalog").write()
# degradeDeepFieldPhotometry.finalize()

### DP 1.3: LSST photometry with no spec-zs

In [13]:
noisyPhotometryPath_WideFastDeep = f"{outputs_directory}//PHOTOMETRY_WideFastDeep_noisy_{date}.pq"

getNoisyWideFastDeepPhotometry = MultiSurveyErrorModel.make_stage(
    name = "getNoisyWideFastDeepPhotometry",
    
    inputType  = inputType,
    outputType = outputType,

    noisy_catalog     = noisyPhotometryPath_WideFastDeep,
    
    m5     = M5_DEPTHS_WideFastDeep,
    bands  = BANDS_WideFastDeep,
    nYrObs = nYrObs,
    nVisYr = nVisYr,
    gamma  = gamma,
    sigLim = sigLim,
    
    seed = seed
)

getNoisyWideFastDeepPhotometry.set_data("noiseless_catalog", PHOTOMETRY_WideFastDeep_noiseless) 
getNoisyWideFastDeepPhotometry.run()
getNoisyWideFastDeepPhotometry.get_handle("noisy_catalog").write()
getNoisyWideFastDeepPhotometry.finalize()

2026-07-03 16:56:04 [info     ] Inserting handle into data store.  noiseless_catalog: None, getNoisyWideFastDeepPhotometry
2026-07-03 16:56:04 [info     ] Inserting handle into data store.  noisy_catalog_getNoisyWideFastDeepPhotometry: /pscratch/sd/s/sajkov/analysis_pipeline_outputs/03Jul26/inprogress_PHOTOMETRY_WideFastDeep_noisy_03Jul26.pq, getNoisyWideFastDeepPhotometry


### Check outputs

In [14]:
PHOTOMETRY_DeepField_noisy    = tables_io.read(f"{outputs_directory}/PHOTOMETRY_DeepField_noisy_03Jul26.pq")
PHOTOMETRY_WideFastDeep_noisy = tables_io.read(f"{outputs_directory}/PHOTOMETRY_WideFastDeep_noisy_03Jul26.pq")

column_list None
column_list None


# Stage 2: etimate photo-zs with LePhare on DP 1.1

In [2]:
from rail.utils.path_utils import RAILDIR

In [3]:
trainFile = os.path.join(RAILDIR, 'rail/examples_data/testdata/output_table_conv_train.hdf5')
testFile = os.path.join(RAILDIR, 'rail/examples_data/testdata/output_table_conv_test.hdf5')
# traindata_io = tables_io.read(trainFile)
# testdata_io = tables_io.read(testFile)

In [5]:
lephare_config_file = os.path.join(RAILDIR, 'rail/examples_data/estimation_data/data/lsst.para')

In [8]:
import shutil
shutil.copy(lephare_config_file, "/pscratch/sd/s/sajkov/analysis_pipeline")

'/pscratch/sd/s/sajkov/analysis_pipeline/lsst.para'

In [19]:
traindata_io.keys()

odict_keys(['mag_err_g_lsst', 'mag_err_i_lsst', 'mag_err_r_lsst', 'mag_err_u_lsst', 'mag_err_y_lsst', 'mag_err_z_lsst', 'mag_g_lsst', 'mag_i_lsst', 'mag_r_lsst', 'mag_u_lsst', 'mag_y_lsst', 'mag_z_lsst', 'redshift'])

In [21]:
testdata_io.keys()

odict_keys(['mag_err_g_lsst', 'mag_err_i_lsst', 'mag_err_r_lsst', 'mag_err_u_lsst', 'mag_err_y_lsst', 'mag_err_z_lsst', 'mag_g_lsst', 'mag_i_lsst', 'mag_r_lsst', 'mag_u_lsst', 'mag_y_lsst', 'mag_z_lsst', 'redshift'])

In [23]:
lephare_config

{'ADAPT_BAND': (ADAPT_BAND, 5),
 'ADAPT_CONTEXT': (ADAPT_CONTEXT, -1),
 'ADAPT_LIM': (ADAPT_LIM, 1.5,23.0),
 'ADAPT_MODBIN': (ADAPT_MODBIN, 1,1000),
 'ADAPT_ZBIN': (ADAPT_ZBIN, 0.01,6),
 'ADDITIONAL_MAG': (ADDITIONAL_MAG, none),
 'ADD_DUSTEM': (ADD_DUSTEM, NO),
 'ADD_EMLINES': (ADD_EMLINES, 0,10000),
 'AGE_RANGE': (AGE_RANGE, 0.,15.e9),
 'AUTO_ADAPT': (AUTO_ADAPT, NO),
 'CAT_FMT': (CAT_FMT, MEME),
 'CAT_IN': (CAT_IN, bidon),
 'CAT_LINES': (CAT_LINES, 0,1000000000),
 'CAT_MAG': (CAT_MAG, AB),
 'CAT_OUT': (CAT_OUT, zphot.out),
 'CAT_TYPE': (CAT_TYPE, LONG),
 'CHI2_OUT': (CHI2_OUT, NO),
 'COSMOLOGY': (COSMOLOGY, 70,0.3,0.7),
 'DZ_WIN': (DZ_WIN, 1.0),
 'EBV_RANGE': (EBV_RANGE, 0,9),
 'EB_V': (EB_V, 0.,0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.5),
 'EM_DISPERSION': (EM_DISPERSION, 0.5,0.75,1.,1.5,2.),
 'EM_LINES': (EM_LINES, EMP_UV),
 'ERR_FACTOR': (ERR_FACTOR, 1.5),
 'ERR_SCALE': (ERR_SCALE, 0.02,0.02,0.02,0.02,0.02,0.02),
 'EXTERNALZ_FILE': (EXTERNALZ_FILE, NONE),
 'EXTINC_LAW': (EXTINC_LAW, 

In [22]:
lephare_config_file = os.path.join(RAILDIR, 'rail/examples_data/estimation_data/data/lsst.para')
lephare_config = lp.read_config(lephare_config_file)

lp.data_retrieval.get_auxiliary_data(keymap=lephare_config)

Number of keywords read in the config file: 86
Registry file downloaded and saved as data_registry.txt.


Created directory: /global/homes/s/sajkov/.cache/lephare/data/vega
Created directory: /global/homes/s/sajkov/.cache/lephare/data/sed/STAR/BD
Created directory: /global/homes/s/sajkov/.cache/lephare/data/filt/lsst
Created directory: /global/homes/s/sajkov/.cache/lephare/data/sed/QSO/SALVATO09
Created directory: /global/homes/s/sajkov/.cache/lephare/data/ext
Created directory: /global/homes/s/sajkov/.cache/lephare/data/examples
Created directory: /global/homes/s/sajkov/.cache/lephare/data/sed/STAR/LAGET
Created directory: /global/homes/s/sajkov/.cache/lephare/data/sed/STAR/PICKLES
Created directory: /global/homes/s/sajkov/.cache/lephare/data/sed/GAL/COSMOS_SED
Created directory: /global/homes/s/sajkov/.cache/lephare/data/sed/GAL/BETHERMIN12
Created directory: /global/homes/s/sajkov/.cache/lephare/data/opa
Created directory: /global/homes/s/sajkov/.cache/lephare/data/sed/STAR/WD
Checking/downloading 421 files...


421 completed.
All files downloaded successfully and are non-empty.


# Stage 3: inform two UMAPs:

### DP 3.1: with photoz-s

### DP 3.2: with spec-zs

# Stage 4: get photo-zs for DP 1.2

### DP 4.1: photo-zs from 3.1

### DP 4.2: photo-zs from 3.2

# Stage 5: compare photo-zs